> 在线讲义：[第 62 章 · 视觉-语言对齐的数学：温度、InfoNCE 下界、SigLIP 的偏置、零样本分类、视觉 token 数账](https://wikieden.github.io/math-and-eval-stats/lessons/62-vl-alignment.html) · 本书《数学与实验统计》

# 第 62 章 · 视觉-语言对齐的数学：温度、InfoNCE 下界、SigLIP 的偏置、零样本分类、视觉 token 数账

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/62-vl-alignment.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 温度 | 损失下限 | `clip_floor(n, tau)` | 8.40 → 1.3e-8 |
| 第二步 互信息下界 | log N − L | `mi_bound(n, loss)` | 8.40 nats |
| 第三步 SigLIP 偏置 | 初始损失 | `siglip_init_loss(n, b)` | 11,355.8 vs 0.744 |
| 第四步 零样本 | 余弦 argmax | `zero_shot(ensemble=...)` | 狗 → 猫 |
| 第五步 token 数 | patch / 合并 | `vit_tokens`、`qwen2vl_tokens` | 576、1536 |

例子：温度 τ 决定 softmax 能有多“尖”（N 个候选时损失的下限）；InfoNCE 给出的互信息下界 log N − L 和它的上限 log N；
SigLIP 初始化 b = −10 的原因（每张图 N − 1 个负样本在初始时的损失）；零样本分类 = 余弦相似度的 argmax（含提示词集成）；
视觉 token 数账（ViT patch、Q-Former、Qwen2-VL 的 2×2 合并）。
里程碑 ME-M36 的“在小批量上实现 CLIP InfoNCE 与 SigLIP 损失，看温度的作用”留给自己做；这里不写整批的损失函数。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np

**函数 `clip_floor`**：余弦相似度在 [−1, 1]：正样本 +1、其余 N − 1 个都是 −1 时，softmax 交叉熵最小也有 log(1 + (N−1)e^{−2/τ})。

In [ ]:
def clip_floor(n: int, tau: float) -> float:
    """余弦相似度在 [−1, 1]：正样本 +1、其余 N − 1 个都是 −1 时，softmax 交叉熵最小也有 log(1 + (N−1)e^{−2/τ})。"""
    return math.log1p((n - 1) * math.exp(-2 / tau))

**函数 `mi_bound`**：InfoNCE 下界：I ≥ log N − L（单位 nats），所以最多只能证明 log N。

In [ ]:
def mi_bound(n: int, loss: float) -> float:
    """InfoNCE 下界：I ≥ log N − L（单位 nats），所以最多只能证明 log N。"""
    return math.log(n) - loss

**函数 `siglip_init_loss`**：一张图对一批 N 段文字：1 个正样本 −log σ(t·cos + b)，N − 1 个负样本 −log σ(−(t·cos + b))。

In [ ]:
def siglip_init_loss(n: int, b: float, t: float = 10.0, cos: float = 0.0) -> tuple[float, float]:
    """一张图对一批 N 段文字：1 个正样本 −log σ(t·cos + b)，N − 1 个负样本 −log σ(−(t·cos + b))。
    初始化时嵌入随机，cos ≈ 0。返回 (正样本项, 负样本项之和)。"""
    z = t * cos + b
    pos = math.log1p(math.exp(-z))
    neg = (n - 1) * math.log1p(math.exp(z))
    return pos, neg

**函数 `normalize`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def normalize(v):
    v = np.asarray(v, float)
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

In [ ]:
IMG = normalize([0.9, 0.3, 0.1])                       # 一张猫的图
PROMPTS = {                                            # 每类两个提示词的文本嵌入
    "猫": normalize([[0.5, 0.8, 0.3], [0.95, 0.0, 0.3]]),
    "狗": normalize([[0.75, 0.6, 0.1], [0.2, 0.9, 0.3]]),
    "车": normalize([[0.1, 0.2, 0.9], [0.0, 0.4, 0.9]]),
}

**函数 `zero_shot`**：每类的文本嵌入（集成 = 先平均再归一化；不集成 = 只用第一个提示词）和图像嵌入的余弦相似度。

In [ ]:
def zero_shot(img=IMG, prompts=PROMPTS, ensemble: bool = True) -> dict[str, float]:
    """每类的文本嵌入（集成 = 先平均再归一化；不集成 = 只用第一个提示词）和图像嵌入的余弦相似度。"""
    out = {}
    for c, e in prompts.items():
        t = normalize(e.mean(0)) if ensemble else e[0]
        out[c] = float(img @ t)
    return out

**函数 `vit_tokens`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def vit_tokens(side: int, patch: int) -> int:
    return (side // patch) ** 2

**函数 `qwen2vl_tokens`**：patch 14、再 2×2 合并 → 每个 token 对应 28×28 像素。

In [ ]:
def qwen2vl_tokens(h: int, w: int) -> int:
    """patch 14、再 2×2 合并 → 每个 token 对应 28×28 像素。"""
    return (h // 28) * (w // 28)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 温度：softmax 能有多尖")
for tau in (1.0, 0.07, 0.01):
    print(f"  N = 32,768、τ = {tau}：损失下限 {clip_floor(32768, tau):.3g}（logit 范围 ±{1 / tau:.1f}）")

In [ ]:
print("\n第二步 InfoNCE 与互信息下界")
for n, loss in ((256, 2.0), (32768, 2.0), (32768, 0.0)):
    print(f"  N = {n:,}、损失 {loss}：I ≥ {mi_bound(n, loss):.2f} nats（上限 log N = {math.log(n):.2f}）")

In [ ]:
print("\n第三步 SigLIP 的偏置初始化（t = 10，初始 cos ≈ 0）")
for b in (0.0, -10.0):
    pos, neg = siglip_init_loss(16384, b)
    print(f"  N = 16,384、b = {b}：正样本项 {pos:.3f}，负样本项合计 {neg:.3f}")

In [ ]:
print("\n第四步 零样本分类 = 余弦相似度")
for ens in (False, True):
    s = zero_shot(ensemble=ens)
    print(f"  {'提示词集成' if ens else '只用一个提示词'}：" + "，".join(f"{c} {v:.4f}" for c, v in s.items()) + f" → {max(s, key=s.get)}")

In [ ]:
print("\n第五步 视觉 token 数账")
print(f"  ViT-L/14 @ 224：{vit_tokens(224, 14)} 个；@ 336（LLaVA-1.5）：{vit_tokens(336, 14)} 个")
print("  BLIP-2 Q-Former：固定 32 个 query")
print(f"  Qwen2-VL 1344×896 的图：{qwen2vl_tokens(896, 1344)} 个；8 帧 448×448 的视频（两帧再合一）：{qwen2vl_tokens(448, 448) * 8 // 2} 个")

## 3. 代码里才看得到的

1. **log1p 不是装饰**：b = −10 时 e^b ≈ 4.5e-5，直接写 `log(1 + e^b)` 在 float32 里会丢精度；`log1p` 才稳。
2. **集成要“先平均再归一化”**：平均后的向量模长小于 1，不归一化的话所有类的分数都被缩小，比较的是错的量。
3. **没写整批的损失函数**：在小批量上实现 CLIP InfoNCE 与 SigLIP 损失、扫温度看效果，留给里程碑 ME-M36。

## 4. 自己改一改

1. 把 `PROMPTS` 里“猫”的第二个提示词改成和第一个一样，集成后还能分对吗？
2. `siglip_init_loss` 里把初始 cos 改成 0.1（嵌入不完全随机），b = −10 时两部分损失怎么变？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 余弦只在正负一，除个小 τ 才够尖；
> InfoNCE 下界 log N 减损失，批小就封顶；
> SigLIP 每对单独判，负多正少偏置设负十；
> 零样本是文本当权重，集成平均提示词；
> token 数决定成本，ImageBind 借图像传递对齐。